# 💻 Laptop Price Predictor — GitHub Dataset

Predict laptop price using Linear Regression with a CSV loaded directly from GitHub.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib flask
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
GITHUB_URL = "https://raw.githubusercontent.com/campusx-official/laptop-price-predictor-regression-project/main/laptop_data.csv"
df = pd.read_csv(GITHUB_URL)
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
df = df.rename(columns={'Company':'Brand','Ram':'RAM','Cpu':'Processor','Memory':'Storage'})
df['RAM_GB'] = df['RAM'].astype(str).str.extract(r'(\d+)')[0].astype(float)

def processor_family(value):
    s = str(value).lower()
    if 'core i9' in s: return 'Intel Core i9'
    if 'core i7' in s: return 'Intel Core i7'
    if 'core i5' in s: return 'Intel Core i5'
    if 'core i3' in s: return 'Intel Core i3'
    if 'core m' in s: return 'Intel Core M'
    if 'celeron' in s: return 'Intel Celeron'
    if 'pentium' in s: return 'Intel Pentium'
    if 'atom' in s: return 'Intel Atom'
    if 'ryzen 7' in s: return 'AMD Ryzen 7'
    if 'ryzen 5' in s: return 'AMD Ryzen 5'
    if 'ryzen 3' in s: return 'AMD Ryzen 3'
    if 'amd' in s: return 'AMD Other'
    return 'Other'

def storage_gb(value):
    values = re.findall(r'(\d+(?:\.\d+)?)\s*(tb|gb)', str(value).lower())
    total = 0
    for number, unit in values:
        total += float(number) * (1000 if unit == 'tb' else 1)
    return total if total > 0 else np.nan

df['Processor_Family'] = df['Processor'].apply(processor_family)
df['Storage_GB'] = df['Storage'].apply(storage_gb)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
data = df[['Brand','RAM_GB','Processor_Family','Storage_GB','Price']].dropna()
display(data.head())


In [ ]:
X = data[['Brand','RAM_GB','Processor_Family','Storage_GB']]
y = data['Price']

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), ['Brand','Processor_Family']),
    ('numeric', SimpleImputer(strategy='median'), ['RAM_GB','Storage_GB'])
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE:', mean_absolute_error(y_test, pred))
print('RMSE:', mean_squared_error(y_test, pred) ** 0.5)
print('R2 Score:', r2_score(y_test, pred))


In [ ]:
print('Enter laptop details')
brand = input('Brand: ')
ram = float(input('RAM (GB): '))
processor = input('Processor Family: ')
storage = float(input('Storage (GB): '))

user_data = pd.DataFrame([{
    'Brand': brand,
    'RAM_GB': ram,
    'Processor_Family': processor,
    'Storage_GB': storage
}])

prediction = max(0, float(model.predict(user_data)[0]))
print(f'Predicted Laptop Price: ₹{prediction:,.0f}')
